# Voces falsifier v3 — separating fragmentation / familiarity / **meaning** / namehood

v2 decomposed the deep-Greek effect into fragmentation + "lexicality" + (null) namehood — but its lexical
cohort was high-frequency international loanwords (ΦΙΛΟΣΟΦΙΑ, ΔΗΜΟΚΡΑΤΙΑ), so "lexicality" confounded **meaning,
frequency, and familiarity**. v3 splits them: a large real-Greek pool, filtered to the voces' Greek-token count
and split by the model's **own surprisal** into FAMILIAR (low-surprisal) vs UNFAMILIAR (high-surprisal) — both
meaningful, differing only in familiarity.

Contrasts (deep Greek-minus-Latin name-likeness, bootstrap 95% CIs):
- **FRAGMENTATION** = asemic@target − asemic@low  (tokenizer)
- **FAMILIARITY**  = lexical_unfamiliar − lexical_familiar  (both meaningful → isolates frequency/familiarity)
- **MEANING**      = asemic − lexical_unfamiliar  (both unfamiliar → isolates meaning)
- **NAMEHOOD**     = voces − asemic  (frag+novelty matched → isolates names)

Also: multi-seed asemic cohorts, per-layer profiles, absolute Greek name-likeness, enlarged name anchor.

**Run:** Runtime → GPU (T4) → Run all. ~20 min on Mistral-7B-v0.3 (4-bit).

In [ ]:
%pip -q install "transformers>=4.44" accelerate bitsandbytes sentencepiece 2>/dev/null
import torch, numpy as np, random
BASE_SEED=0; np.random.seed(BASE_SEED); random.seed(BASE_SEED); torch.manual_seed(BASE_SEED)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.3"   # or "google/gemma-2-9b" (+HF_TOKEN in Colab Secrets)
LOAD_4BIT  = True
N_SEEDS    = 5         # asemic/random cohorts regenerated across this many seeds
VERSION    = "falsifier-v3-separation_" + MODEL_NAME.split("/")[-1]
import os
HF_TOKEN = os.environ.get("HF_TOKEN","")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or (userdata.get("HF_TOKEN") or "")
except Exception: pass
if HF_TOKEN:
    from huggingface_hub import login; login(HF_TOKEN); print("HF: logged in")
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token
kw = dict(torch_dtype=torch.float16, device_map="auto", output_hidden_states=True, low_cpu_mem_usage=True)
if "gemma-2" in MODEL_NAME.lower(): kw["attn_implementation"]="eager"
if LOAD_4BIT:
    kw["quantization_config"]=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw).eval()
N_LAYERS = model.config.num_hidden_layers
DEV = next(model.parameters()).device
print("loaded:", MODEL_NAME, "| layers:", N_LAYERS, "| device:", DEV)

In [ ]:
# transliterator + extraction + surprisal (carrier frame identical to the cross-family notebook)
_DIGRAPHS=[("TH","Θ"),("PH","Φ"),("CH","Χ"),("PS","Ψ"),("NG","ΝΓ")]
_SINGLES={"A":"Α","B":"Β","G":"Γ","D":"Δ","E":"Ε","Z":"Ζ","H":"Η","I":"Ι","K":"Κ","L":"Λ","M":"Μ",
          "N":"Ν","X":"Ξ","O":"Ο","P":"Π","R":"Ρ","S":"Σ","T":"Τ","U":"Υ","Y":"Υ","W":"Ω"}
def to_greek(s):
    s=s.upper()
    for a,b in _DIGRAPHS: s=s.replace(a,b)
    return "".join(_SINGLES.get(c,c) for c in s)
def ngreek_tok(latin): return len(tok.encode(to_greek(latin), add_special_tokens=False))

FRAME_PRE="The string "; FRAME_POST=" is written on the page."
@torch.no_grad()
def extract_reps(s):
    pre_ids=tok(FRAME_PRE, add_special_tokens=True).input_ids
    full_ids=tok(FRAME_PRE+s, add_special_tokens=True).input_ids
    start=len(pre_ids); end=len(full_ids)
    ids=tok(FRAME_PRE+s+FRAME_POST, return_tensors="pt", add_special_tokens=True).input_ids.to(DEV)
    end=min(end, ids.shape[1])
    if end<=start: start=max(0,end-1)
    hs=model(ids).hidden_states
    return np.stack([h[0,start:end,:].float().mean(0).cpu().numpy() for h in hs])
def build_matrix(strings): return np.stack([extract_reps(s) for s in strings])

@torch.no_grad()
def mean_surprisal(greek_str):
    # model's own mean per-token surprisal (nats) for the Greek string in the carrier frame -> familiarity proxy
    ids=tok(FRAME_PRE+greek_str+FRAME_POST, return_tensors="pt", add_special_tokens=True).input_ids.to(DEV)
    pre=len(tok(FRAME_PRE, add_special_tokens=True).input_ids)
    full=len(tok(FRAME_PRE+greek_str, add_special_tokens=True).input_ids)
    logits=model(ids).logits[0]
    logp=torch.log_softmax(logits[:-1].float(), -1)
    tgt=ids[0,1:]
    nll=-logp[torch.arange(tgt.shape[0]), tgt]
    span=[i-1 for i in range(pre, min(full, ids.shape[1]))]
    return float(nll[span].mean().cpu()) if span else float('nan')
print("harness + surprisal ready.")

In [ ]:
# anchors (enlarged name set ~32 to reduce centroid noise) + voces
NAMES_LAT=["AGAMEMNON","ODYSSEUS","ACHILLEUS","PENELOPE","ALEXANDROS","SOKRATES","ARISTOTELES","HERAKLES",
 "PERSEPHONE","APHRODITE","POSEIDON","ARTEMIS","DIONYSOS","ASKLEPIOS","HEPHAISTOS","KLEOPATRA","AGAMEMNON",
 "MENELAOS","AGATHON","THEMISTOKLES","PERIKLES","DEMOSTHENES","EURIPIDES","SOPHOKLES","AISCHYLOS","HERODOTOS",
 "THOUKYDIDES","PYTHAGORAS","ANAXAGORAS","EMPEDOKLES","PARMENIDES","ANTIGONE"]
NAMES_LAT=list(dict.fromkeys(NAMES_LAT))  # dedupe, keep order
_CONS=list("BGDZKLMNPRSTFCH"); _VOW=list("AEIOUY")
VOCES_LAT=["ABLANATHANALBA","AKRAMMACHAMAREI","SESENGENBARPHARANGES","SEMESILAM","SEMESEILAMPS","MASKELLI",
 "MASKELLO","NEBOUTOSOUALETH","DAMNAMENEUS","ASKION","KATASKION","LIXTETRAX","AISION","BAINCHOOOCH","THOBARRABAU",
 "CHABRACH","PHNESCHER","BORKA","PHORBA","AROURARELYOTH","KMEPH","THENOR","AEEIOYO","IEOUAEOI","OREOBAZAGRA",
 "PROTOPHANES","NEPHERIERI","AKTIOPHI","ORORIOUTH","BAKAXICHYCH","ABERAMENTHOOU","LERTHEXANAX","SOROORMERPHERGAR",
 "PATATHNAX","IARBATHA","SANKANTHARA","PSINOTHER","CHTHETHONI","MENEBAINCHUCH","BARZAN","SISISRO","THENOB",
 "ARSENOPHRE","BACHUCH","PHORPHORBA","HARMONTHARTHOCHE","NEOPHOXOTHA","ARISTANABAZAO","PCHORBAZANACHU",
 "ZALAMOIRLALITH","EILESILARMU","TANTINURACHTH","CHORCHORNATHI","PHANTHENPHYPHLIA","AZAZAEISTHAILICH",
 "MENNYTHYTH","SERYCHARRALMIO","AKHEBUKROM"]
THEONYM_ROOTS=["IAO","SABAOTH","BAOTH","AOTH","ADONAI","ELOAI","ELOI","ABRASAX","ABRAXAS","ABRA","MICHA",
 "ERESHKIGAL","SABA","ADON","OTH","IAE","IEOU","SETH","HOR","OSIR","ISIS","AMOUN","THOTH"]
def contamination_T(s):
    s=s.upper(); cov=[False]*len(s)
    for r in THEONYM_ROOTS:
        st=0
        while True:
            i=s.find(r,st)
            if i<0: break
            for j in range(i,i+len(r)): cov[j]=True
            st=i+1
    return sum(cov)/max(1,len(s))
VOCES_LOWT=[v for v in VOCES_LAT if contamination_T(v)<0.15]
TARGET=int(round(np.median([ngreek_tok(v) for v in VOCES_LOWT])))
print(f"name anchor n={len(NAMES_LAT)} | voces low-T n={len(VOCES_LOWT)} | TARGET Greek tok={TARGET}")

In [ ]:
# REAL-GREEK pool (broad: loanwords + rare/archaic/technical/Homeric) -> split by surprisal into familiar/unfamiliar
REAL_GREEK_POOL=[
 # familiar international/learned loanwords
 "PHILOSOPHIA","DEMOKRATIA","ANTHROPOLOGIA","BIBLIOTHEKE","KATASTROPHE","METAMORPHOSIS","OIKONOMIA","GEOGRAPHIA",
 "ASTRONOMIA","MATHEMATIKE","THEOLOGIA","PSYCHOLOGIA","TECHNOLOGIA","ENKYKLOPAIDEIA","ARISTOKRATIA","KOSMOLOGIA",
 "SYMPHONIA","EUCHARISTIA","DIALEKTIKE","ETYMOLOGIA",
 # rarer / archaic / Homeric / technical (meaningful but lower-frequency)
 "GLAUKOPIS","RHODODAKTYLOS","ERIGDOUPOS","KUDOIMOS","AMPHIGUEEIS","ENOSICHTHON","NEPHELEGERETA","ARGEIPHONTES",
 "POLUPHLOISBOS","LEUKOLENOS","BOOPIS","KHALKEOPHONOS","EUKNEMIDES","PODARKES","KORUTHAIOLOS","OBRIMOPATRE",
 "TANUPEPLOS","KALLIPAREOS","MELIEDES","PERIKALLES","AMPHIELISSA","EUPLOKAMOS","PHILOMMEIDES","TERPSIMBROTOS",
 "CHRUSOTHRONOS","BATHUKOLPOS","AIGIOCHOS","ERIBROMOS","PALAIONTOLOGIA","HERMENEUTIKE","PARTHENOGENESIS"]
REAL_GREEK_POOL=list(dict.fromkeys(REAL_GREEK_POOL))
pool=[w for w in REAL_GREEK_POOL if abs(ngreek_tok(w)-TARGET)<=2]
print(f"real-Greek pool near {TARGET}±2 tok: {len(pool)}")
# surprisal-split: compute mean surprisal of each (Greek form), split at the median
surp={w: mean_surprisal(to_greek(w)) for w in pool}
med=float(np.median(list(surp.values())))
LEX_FAMILIAR  =sorted([w for w in pool if surp[w]< med], key=lambda w:surp[w])       # low surprisal = familiar
LEX_UNFAMILIAR=sorted([w for w in pool if surp[w]>=med], key=lambda w:-surp[w])      # high surprisal = unfamiliar
print(f"familiar (low-surp) n={len(LEX_FAMILIAR)}: {LEX_FAMILIAR[:6]}")
print(f"unfamiliar (high-surp) n={len(LEX_UNFAMILIAR)}: {LEX_UNFAMILIAR[:6]}")
print(f"median surprisal split = {med:.2f} nats | familiar mean {np.mean([surp[w] for w in LEX_FAMILIAR]):.2f} vs unfamiliar {np.mean([surp[w] for w in LEX_UNFAMILIAR]):.2f}")

In [ ]:
# asemic + lowfrag cohorts; asemic regenerated across N_SEEDS (robustness to the draw)
def gen_asemic(target, n, seed, tol=1, mt=40000):
    rng=random.Random(seed); out=[]; t=0
    while len(out)<n and t<mt:
        t+=1; L=rng.randint(6,16); s="".join(rng.choice(_CONS+_VOW) for _ in range(L))
        if abs(ngreek_tok(s)-target)<=tol: out.append(s)
    return out
def gen_random_anchor(n, seed):
    rng=random.Random(seed+999); return ["".join(rng.choice(_CONS) for _ in range(rng.randint(5,11))) for _ in range(n)]
ASEMIC_BY_SEED={s: gen_asemic(TARGET, 28, s) for s in range(N_SEEDS)}
RANDOM_BY_SEED={s: gen_random_anchor(30, s) for s in range(N_SEEDS)}
NONNAME_LOWFRAG=["EN","DYO","TRIA","TESSARA","PENTE","EX","EPTA","OKTO","ENNEA","DEKA","KAI","ALLA","OUTOS",
 "GAR","MEN","EIS","EK","PROS","META","PERI","BIBLION","KARDIA","LOGOS","ERGON","TOPOS","PHONE"]
def frag(c): a=np.array([ngreek_tok(x) for x in c]); return a.mean()
print("cohort Greek tok/str (mean):")
for nm,c in [("voces",VOCES_LOWT),("asemic(seed0)",ASEMIC_BY_SEED[0]),("lex_familiar",LEX_FAMILIAR),
             ("lex_unfamiliar",LEX_UNFAMILIAR),("lowfrag",NONNAME_LOWFRAG)]:
    print(f"  {nm:16s} {frag(c):5.2f}  (n={len(c)})")

In [ ]:
# extract reps; deterministic cohorts once, seed-varying cohorts per seed
def reps_both(latins): return {"latin":build_matrix(latins), "greek":build_matrix([to_greek(x) for x in latins])}
import time; t0=time.time()
DET={k:reps_both(v) for k,v in {"name":NAMES_LAT,"voces":VOCES_LOWT,"lex_familiar":LEX_FAMILIAR,
                                 "lex_unfamiliar":LEX_UNFAMILIAR,"lowfrag":NONNAME_LOWFRAG}.items()}
ASEMIC_REPS={s:reps_both(ASEMIC_BY_SEED[s]) for s in range(N_SEEDS)}
RANDOM_REPS={s:reps_both(RANDOM_BY_SEED[s]) for s in range(N_SEEDS)}
print("extraction done in", round(time.time()-t0,1),"s")

In [ ]:
# name-likeness machinery + per-layer profiles + absolute Greek name-likeness
DEEP=range(int(0.5*N_LAYERS), N_LAYERS+1)
def _nrm(x): return x/(np.linalg.norm(x)+1e-9)
def centroids(script, seed):
    nameC=DET["name"][script]; randC=RANDOM_REPS[seed][script]
    return nameC, randC
def nl_profile(A, script, seed):
    # per-layer name-likeness for matrix A [n,L+1,d], vs name/random centroids of this script+seed
    nameC,randC=centroids(script,seed)
    prof=[]
    for L in range(N_LAYERS+1):
        nc=_nrm(nameC[:,L,:].mean(0)); rc=_nrm(randC[:,L,:].mean(0))
        prof.append(np.array([_nrm(A[i,L,:])@nc - _nrm(A[i,L,:])@rc for i in range(A.shape[0])]))
    return prof  # list over layers of per-string arrays
def deep_gap_perstring(reps, seed):
    gl=nl_profile(reps["greek"], "greek", seed); ll=nl_profile(reps["latin"], "latin", seed)
    n=reps["greek"].shape[0]
    return np.array([np.mean([gl[L][i]-ll[L][i] for L in DEEP]) for i in range(n)])
def abs_greek_deep(reps, seed):
    gl=nl_profile(reps["greek"], "greek", seed); n=reps["greek"].shape[0]
    return np.array([np.mean([gl[L][i] for L in DEEP]) for i in range(n)])
def boot(ps,B=2000,seed=0):
    rng=np.random.RandomState(seed); n=len(ps); m=[ps[rng.randint(0,n,n)].mean() for _ in range(B)]
    return float(ps.mean()), float(np.percentile(m,2.5)), float(np.percentile(m,97.5))

# pool asemic across seeds for the headline (and keep per-seed for stability)
asemic_ps_pooled=np.concatenate([deep_gap_perstring(ASEMIC_REPS[s], s) for s in range(N_SEEDS)])
asemic_perseed=[float(deep_gap_perstring(ASEMIC_REPS[s], s).mean()) for s in range(N_SEEDS)]
GAPS={}
GAPS["voces"]          = boot(deep_gap_perstring(DET["voces"], 0))
GAPS["asemic_pooled"]  = boot(asemic_ps_pooled)
GAPS["lex_familiar"]   = boot(deep_gap_perstring(DET["lex_familiar"], 0))
GAPS["lex_unfamiliar"] = boot(deep_gap_perstring(DET["lex_unfamiliar"], 0))
GAPS["lowfrag"]        = boot(deep_gap_perstring(DET["lowfrag"], 0))
print("deep Greek-minus-Latin gap [95% CI]:")
for k,(m,lo,hi) in GAPS.items(): print(f"  {k:16s} {m:+.4f} [{lo:+.4f},{hi:+.4f}]")
print(f"asemic per-seed deep gaps: {[round(x,4) for x in asemic_perseed]}  (sd={np.std(asemic_perseed):.4f}) -> draw-robustness")
print("absolute Greek deep name-likeness (no Latin subtraction):")
for k,reps in [("voces",DET["voces"]),("lex_familiar",DET["lex_familiar"]),("lex_unfamiliar",DET["lex_unfamiliar"])]:
    print(f"  {k:16s} {abs_greek_deep(reps,0).mean():+.4f}")

In [ ]:
# factorial with CI-based verdicts (no canned null-affirmation)
def diff_ci(psa,psb,B=2000,seed=1):
    rng=np.random.RandomState(seed)
    d=[psa[rng.randint(0,len(psa),len(psa))].mean()-psb[rng.randint(0,len(psb),len(psb))].mean() for _ in range(B)]
    return float(psa.mean()-psb.mean()), float(np.percentile(d,2.5)), float(np.percentile(d,97.5))
PS={"voces":deep_gap_perstring(DET["voces"],0),"asemic":asemic_ps_pooled,
    "lex_familiar":deep_gap_perstring(DET["lex_familiar"],0),"lex_unfamiliar":deep_gap_perstring(DET["lex_unfamiliar"],0),
    "lowfrag":deep_gap_perstring(DET["lowfrag"],0)}
def verdict(name,a,b):
    d,lo,hi=diff_ci(PS[a],PS[b]); sig=(lo>0 or hi<0)
    tag = ("SIGNIFICANT +" if d>0 else "SIGNIFICANT -") if sig else "n.s. (CI spans 0 -> NOT evidence for the null, just no detected diff)"
    print(f"[{name:14s}] {a} - {b} = {d:+.4f} [{lo:+.4f},{hi:+.4f}]  {tag}")
    return dict(a=a,b=b,diff=d,ci_lo=lo,ci_hi=hi,significant=bool(sig))
print("="*78); print("FACTORIAL — model:", MODEL_NAME); print("="*78)
R={}
R["FRAGMENTATION"]= verdict("FRAGMENTATION","asemic","lowfrag")       # tokenizer
R["FAMILIARITY"]  = verdict("FAMILIARITY","lex_unfamiliar","lex_familiar")  # both meaningful -> isolates familiarity
R["MEANING"]      = verdict("MEANING","asemic","lex_unfamiliar")      # both unfamiliar -> isolates meaning
R["NAMEHOOD"]     = verdict("NAMEHOOD","voces","asemic")              # frag+novelty matched -> isolates names
print("\n--- how to read (the bundle v2 could not split) ---")
print("FAMILIARITY sig+ & MEANING n.s. -> the 2nd factor is FAMILIARITY/FREQUENCY, not meaning")
print("MEANING sig+ & FAMILIARITY n.s. -> the 2nd factor is MEANING itself")
print("both sig+                       -> meaning AND familiarity each contribute")

In [ ]:
import json
def fragstat(c): a=np.array([ngreek_tok(x) for x in c]); return dict(mean=float(a.mean()),median=float(np.median(a)),n=len(a))
out=dict(model=MODEL_NAME, version=VERSION, quantized=bool(LOAD_4BIT), n_layers=int(N_LAYERS),
    n_seeds=N_SEEDS, target_greek_tok=TARGET,
    cohort_fragmentation={k:fragstat(c) for k,c in [("voces",VOCES_LOWT),("asemic_seed0",ASEMIC_BY_SEED[0]),
        ("lex_familiar",LEX_FAMILIAR),("lex_unfamiliar",LEX_UNFAMILIAR),("lowfrag",NONNAME_LOWFRAG)]},
    surprisal_split=dict(median_nats=med,
        familiar_mean=float(np.mean([surp[w] for w in LEX_FAMILIAR])),
        unfamiliar_mean=float(np.mean([surp[w] for w in LEX_UNFAMILIAR]))),
    deep_gaps={k:dict(mean=m,ci_lo=lo,ci_hi=hi) for k,(m,lo,hi) in GAPS.items()},
    asemic_perseed_deepgap=asemic_perseed,
    absolute_greek_deep={k:float(abs_greek_deep(DET[k],0).mean()) for k in ["voces","lex_familiar","lex_unfamiliar","lowfrag"]},
    factorial=R,
    layer_profile_greek_minus_latin={k:[float(np.mean([nl_profile(DET[k]["greek"],"greek",0)[L]-nl_profile(DET[k]["latin"],"latin",0)[L]]))
                                        for L in range(N_LAYERS+1)] for k in ["voces","lex_familiar","lex_unfamiliar","lowfrag"]})
fn=f"voces_{VERSION}_results.json"
json.dump(out, open(fn,"w"), indent=2)
print(json.dumps({k:out[k] for k in ["factorial","surprisal_split","asemic_perseed_deepgap"]}, indent=2))
print("\nsaved:", fn)
try:
    from google.colab import files; files.download(fn)
except Exception: pass